# SL-7 Cross-Model Run — Gemma-3-4B-it (T4-safe FP32, family-gated)

Model: `google/gemma-3-4b-it`

This notebook reruns the **unchanged frozen SL-7 v16 benchmark**. The previous FP16 Gemma run produced empty generations on every item; Gemma 3 is known to be numerically unstable in FP16. This version therefore uses the original Gemma weights in **FP32** rather than quantizing the model.

### Hardware strategy
- If Kaggle exposes **2× T4**, the FP32 model is automatically sharded across both GPUs.
- If only **1× T4** is available, the notebook automatically CPU-offloads part of the FP32 model.
- No benchmark image, prompt, answer key, family threshold, or decoding rule is changed.

### Important
Use a fresh Kaggle session. **2× T4 is strongly preferred** because FP32 is slower and larger than FP16.


In [ ]:
%pip install -q -U "transformers==5.17.0" "accelerate>=1.0" "huggingface_hub>=0.30" "pillow" "psutil"


In [ ]:
PROTOCOL_VERSION="SL7_CROSS_MODEL_FAMILY_GATED_V2_GEMMA_FP32"
print("="*82)
print(PROTOCOL_VERSION)
print("Model: google/gemma-3-4b-it")
print("Precision: FP32 (Gemma 3 FP16 empty-output workaround)")
print("Calibration failures gate only dependent families; holdout continues.")
print("="*82)


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import glob, csv, json, time, platform, sys, zipfile, importlib, random, gc
from pathlib import Path
from PIL import Image
import torch, transformers

MAX_NEW_TOKENS=64
INFERENCE_SEED=20261004
random.seed(INFERENCE_SEED); torch.manual_seed(INFERENCE_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(INFERENCE_SEED)

hits=[]
for p in glob.glob("/kaggle/input/**/dataset/items.csv", recursive=True):
    root=Path(p).parent.parent
    if (root/"VERSION_SL7_v16.txt").exists() and (root/"eval/score.py").exists(): hits.append(root)
if not hits: raise FileNotFoundError("Add the snakes-ladders-SL7-multimodel-v1 package as a Kaggle input dataset.")
ROOT=hits[0]
ITEMS=list(csv.DictReader(open(ROOT/"dataset/items.csv",encoding="utf-8")))
CAL=[x for x in ITEMS if x["part"]=="CAL"]
SCORED=[x for x in ITEMS if x["part"]=="SCORED"]
print("Frozen project root:",ROOT)
print("Calibration:",len(CAL),"Scored:",len(SCORED))
sys.path.insert(0,str(ROOT/"eval"))
import score
importlib.reload(score)


In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import psutil

MODEL_ID='google/gemma-3-4b-it'
MODEL_SLUG='gemma3_4b_fp32'
MODEL_DISPLAY='Gemma-3-4B-it'
OUT=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}")
OUT.mkdir(exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator.")

# Hugging Face authentication.
token=os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token=None
if not token:
    raise RuntimeError("HF_TOKEN not found. Add a Kaggle secret named HF_TOKEN and accept the Gemma license on Hugging Face.")
from huggingface_hub import login
login(token=token,add_to_git_credential=False)
print("Hugging Face authentication succeeded.")

# Gemma 3 is numerically unsafe in FP16. Use the original weights in FP32.
# Prefer sharding across two T4 GPUs; fall back to GPU+CPU offload on one T4.
ngpu=torch.cuda.device_count()
cpu_gib=max(8,int(psutil.virtual_memory().available/2**30)-4)
if ngpu>=2:
    max_memory={0:"11GiB",1:"11GiB","cpu":f"{cpu_gib}GiB"}
    print("Detected",ngpu,"GPUs. Using FP32 multi-GPU sharding with max_memory:",max_memory)
else:
    max_memory={0:"8GiB","cpu":f"{cpu_gib}GiB"}
    print("Detected one GPU. Using FP32 GPU+CPU offload with max_memory:",max_memory)

OFFLOAD_DIR="/kaggle/working/gemma3_fp32_offload"
os.makedirs(OFFLOAD_DIR,exist_ok=True)
model=Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR,
    offload_state_dict=True,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
processor=AutoProcessor.from_pretrained(MODEL_ID)

print("hf_device_map:",getattr(model,"hf_device_map",None))
for i in range(torch.cuda.device_count()):
    free,total=torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {(total-free)/2**30:.2f} GiB used / {total/2**30:.2f} GiB total")

env={
 "model_key":MODEL_SLUG,"model_display":MODEL_DISPLAY,"model_id":MODEL_ID,
 "parameters":sum(p.numel() for p in model.parameters()),"transformers":transformers.__version__,
 "torch":torch.__version__,"cuda":torch.version.cuda,
 "gpus":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
 "python":platform.python_version(),"dtype":"float32","attention":"sdpa",
 "decoding":"greedy","max_new_tokens":MAX_NEW_TOKENS,"inference_seed":INFERENCE_SEED,
 "dataset":"SL7_v16_frozen","device_map":getattr(model,"hf_device_map",None),
 "reason_for_fp32":"Gemma3 FP16 can produce empty generations due to numerical overflow"
}
(OUT/"environment.json").write_text(json.dumps(env,indent=2,default=str))
print(json.dumps(env,indent=2,default=str))


In [ ]:
# Accelerate dispatch uses the first CUDA device as the execution entry point.
ENTRY_DEVICE=torch.device("cuda:0")

def run_item(it, debug=False):
    content=[]
    for rel in [p for p in it["image_paths"].split("|") if p]:
        p=str(ROOT/rel)
        if not Path(p).exists(): raise FileNotFoundError(p)
        content.append({"type":"image","path":p})
    content.append({"type":"text","text":it["prompt"]})
    messages=[{"role":"user","content":content}]
    t0=time.time()
    inputs=processor.apply_chat_template(
        messages,add_generation_prompt=True,tokenize=True,
        return_dict=True,return_tensors="pt"
    )
    # Keep integer token tensors integer; cast only floating multimodal tensors to FP32.
    inputs=inputs.to(ENTRY_DEVICE)
    for k,v in list(inputs.items()):
        if torch.is_tensor(v) and torch.is_floating_point(v):
            inputs[k]=v.to(dtype=torch.float32)
    inputs.pop("token_type_ids",None)
    input_len=inputs["input_ids"].shape[-1]
    try:
        with torch.inference_mode():
            generated=model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )
        continuation=generated[0,input_len:]
        reply=processor.decode(continuation,skip_special_tokens=True).strip()
        if debug:
            print("input_len:",input_len,"generated_len:",generated.shape[-1],"continuation_len:",continuation.shape[-1])
            print("reply:",repr(reply))
        return {"item_id":it["item_id"],"raw_output":reply,"seconds":round(time.time()-t0,2)}
    finally:
        gc.collect()
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i): torch.cuda.empty_cache()


In [ ]:
RAW=OUT/"raw_outputs.csv"
outrows=[]
def save_raw():
    with open(RAW,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["item_id","raw_output","seconds"])
        w.writeheader(); w.writerows(outrows)

# PRE-FLIGHT: catch Gemma empty-generation/dtype problems before spending hours.
print("Running one-item Gemma FP32 pre-flight...")
first = run_item(CAL[0], debug=True)
if not first["raw_output"].strip():
    raise RuntimeError(
        "Gemma pre-flight still produced an empty reply. Stop here; do NOT run the 150-item holdout."
    )
print("Pre-flight PASSED:", CAL[0]["item_id"], "=>", first["raw_output"][:160])
outrows.append(first); save_raw()

for idx,it in enumerate(CAL[1:],2):
    outrows.append(run_item(it)); save_raw()
    print(f"CAL {idx}/{len(CAL)}",it["item_id"],"=>",outrows[-1]["raw_output"][:160])

pred={r["item_id"]:r["raw_output"] for r in outrows}
groups={}
for it in CAL:
    parsed=score.parse_strict(pred[it["item_id"]],it["answer_format"])
    ok=parsed.lower()==it["correct_answer"].lower() if parsed else False
    groups.setdefault(it["task_family"],[]).append(ok)
cal_counts={k:(sum(v),len(v)) for k,v in groups.items()}
requirements={"CAL_number":7,"CAL_endpoint":3,"CAL_die":4,"CAL_multi_image":3}
cal_pass={k:(sum(groups.get(k,[]))>=need) for k,need in requirements.items()}
failed=[k for k,v in cal_pass.items() if not v]
family_gate_requirements={
 "B1_zigzag_direction":["CAL_number"],
 "B2_hidden_reconstruction":["CAL_number"],
 "B3_endpoint_anchoring":["CAL_endpoint"],
 "B4_perception_to_action":["CAL_number","CAL_endpoint"],
 "B5_multi_image_position":["CAL_number","CAL_endpoint","CAL_multi_image"],
 "B6_conditioned_counting":["CAL_number","CAL_endpoint"],
 "B7_multi_source_composition":["CAL_number","CAL_die","CAL_multi_image"],
}
family_eligibility={fam:all(cal_pass[g] for g in gates) for fam,gates in family_gate_requirements.items()}
record={"counts":{k:list(v) for k,v in cal_counts.items()},"requirements":requirements,
        "calibration_pass":cal_pass,"original_global_gate_passed":not failed,
        "failed_calibrations":failed,"cross_model_protocol":"family_specific_calibration_gating_v2",
        "family_gate_requirements":family_gate_requirements,"family_eligibility":family_eligibility}
(OUT/"calibration_status.json").write_text(json.dumps(record,indent=2))
print("\nCalibration:",cal_counts); print("Calibration pass map:",cal_pass)
print("\nFamily eligibility:")
for fam,ok in family_eligibility.items(): print(" ",fam,":","ELIGIBLE" if ok else "INELIGIBLE / diagnostic only",family_gate_requirements[fam])
if failed:
    print("\nOriginal global gate would fail for:",failed)
    print("FAMILY-GATED MODE: continuing to the frozen 150-item holdout.")
else:
    print("\nAll calibration gates passed; all seven families are eligible.")


In [ ]:
for idx,it in enumerate(SCORED,1):
    outrows.append(run_item(it)); save_raw()
    if idx%10==0 or idx==len(SCORED): print(f"SCORED {idx}/{len(SCORED)} complete")
print("Raw outputs saved:",RAW)


In [ ]:
# Zero-eligible-family-safe presentation patch; scoring rules are unchanged.
_original_wilson=score.wilson
def _wilson_zero_safe(k,n,*args,**kwargs):
    if n==0: return (float("nan"),float("nan"))
    return _original_wilson(k,n,*args,**kwargs)
score.wilson=_wilson_zero_safe
scored_rows,family_rows,original_cal_pass=score.score(ROOT/"dataset/items.csv",RAW,OUT)
_summary=OUT/"summary.md"
if _summary.exists():
    t=_summary.read_text(encoding="utf-8").replace("nan–nan%","N/A").replace("nan-nan%","N/A")
    _summary.write_text(t,encoding="utf-8")
fam_rows=list(csv.DictReader(open(OUT/"family_summary.csv",encoding="utf-8")))
for r in fam_rows:
    fam=r["family"]; eligible=bool(family_eligibility.get(fam,False)); frozen=(r["confirmed"]=="1")
    r["calibration_eligible"]="1" if eligible else "0"
    r["headline_confirmed"]="1" if (eligible and frozen) else "0"
    reqs=family_gate_requirements.get(fam,[]); failed_reqs=[g for g in reqs if not cal_pass.get(g,False)]
    r["eligibility_note"]="eligible" if eligible else "failed calibration: "+",".join(failed_reqs)
fg=OUT/"family_summary_family_gated.csv"
with open(fg,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=list(fam_rows[0].keys())); w.writeheader(); w.writerows(fam_rows)
L=["# SL-7 Cross-Model Family-Gated Summary","",f"Model: **{MODEL_DISPLAY}**","",
   "Frozen SL-7 v16 stimuli and family thresholds are unchanged. Calibration eligibility is applied per dependent family.","",
   "## Calibration","", "| Calibration | Correct | Required | Pass |","|---|---:|---:|---|"]
for name in ["CAL_number","CAL_endpoint","CAL_die","CAL_multi_image"]:
    k,n=cal_counts.get(name,(0,0)); need=requirements[name]
    L.append(f"| {name.replace('CAL_','')} | {k}/{n} | {need}/{n} | {'PASS' if cal_pass[name] else 'FAIL'} |")
L += ["","## Family results","",
      "| Family | Eligible? | Frozen confirmed? | Headline confirmed? | Control/component | Probe | Gap | Note |",
      "|---|---|---|---|---:|---:|---:|---|"]
for r in fam_rows:
    elig="YES" if r["calibration_eligible"]=="1" else "NO"; conf="YES" if r["confirmed"]=="1" else "NO"; head="YES" if r["headline_confirmed"]=="1" else "NO"
    L.append(f"| {r['family']} | {elig} | {conf} | **{head}** | {r['control_or_component_correct']}/{r['control_or_component_n']} | {r['probe_correct']}/{r['probe_n']} | {r['gap_percentage_points']} pp | {r['eligibility_note']} |")
L += ["","`headline_confirmed=1` requires both the frozen family criterion and all directly relevant calibration prerequisites.",""]
(OUT/"summary_family_gated.md").write_text("\n".join(L),encoding="utf-8")
print((OUT/"summary_family_gated.md").read_text())
zip_path=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}_family_gated.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.name)
print("Download:",zip_path)
